# LUMIERE external OOD evaluation — Pinaya FT vs KL=1e-6 overfit

Side-by-side OOD evaluation of the **two preprint models** on the LUMIERE
external test set (one session per patient, frozen split).

**Pipeline**
1. Load the frozen LUMIERE test split (produced by
   [scripts/freeze_lumiere_test_split.py](freeze_lumiere_test_split.py)).
2. For each model (`pinaya_ft`, `kl1e6_overfit`) and each CFG value, run
   the offline sampler from [scripts/offline_sample_stage2_compare.py](offline_sample_stage2_compare.py)
   end-to-end (re-uses every helper that the in-distribution sweep notebooks
   use, so apples-to-apples).
3. Compute SSIM, LPIPS, W1 inside the expert mask, and FID across all
   four available backbones.
4. Aggregate paired statistics: Wilcoxon signed-rank (Pinaya FT vs KL1e6)
   on per-case metrics, with Cliff's δ effect size.
5. Compare to the in-distribution numbers cached under
   `runs/<model>/data/cfg_sweep_*/ssim_vs_cfg.csv` etc. to compute the OOD
   degradation Δ per modality, per model.

All paths are env-var driven so this notebook is the same code on Gadi and
locally. See the **Parameters** cell below.

## 1. Parameters

On Gadi set these in the shell that started Jupyter (or override them in
this cell):

```bash
export T2G_REPO_DIR=$HOME/text2glioma
export T2G_LUMIERE_DATALIST=/g/data/vp06/$USER/text2glioma_train/data/lumiere_ingested/datalist_lumiere_test.json
export T2G_HF_CACHE=/g/data/vp06/$USER/text2glioma_train/runs/cache/huggingface/hub
export T2G_OUTPUT_DIR=/g/data/vp06/$USER/text2glioma_train/runs/lumiere_eval

# Pinaya FT checkpoints
export T2G_PINAYA_STAGE1_CFG=$T2G_REPO_DIR/configs/stage1_pinaya_decoder_only.yaml
export T2G_PINAYA_LDM_CFG=$T2G_REPO_DIR/configs/ldm_radbert_pinaya_decoder_only.yaml
export T2G_PINAYA_STAGE1_URI=/g/data/vp06/$USER/text2glioma_train/runs/pinaya_decoder_only_v5_no_disc/autoencoder_stage1/final_model.pth
export T2G_PINAYA_LDM_CKPT=/g/data/vp06/$USER/text2glioma_train/runs/pinaya_decoder_only_v5_no_disc/ldm_stage2/best_model.pth

# KL=1e-6 overfit checkpoints
export T2G_KL1E6_STAGE1_CFG=$T2G_REPO_DIR/configs/stage1.yaml
export T2G_KL1E6_LDM_CFG=$T2G_REPO_DIR/configs/ldm_radbert.yaml
export T2G_KL1E6_STAGE1_URI=/g/data/vp06/$USER/text2glioma_train/runs/stage1_overfit_ablate_kl1e6/output/models/best_model.pth
export T2G_KL1E6_LDM_CKPT=/g/data/vp06/$USER/text2glioma_train/runs/ldm_radbert_kl1e6/output/models/best_model.pth
```

In [ ]:
import os, sys, json, importlib
from pathlib import Path

REPO_DIR = Path(os.environ.get("T2G_REPO_DIR", Path.cwd().parent)).resolve()
SCRIPTS_DIR = REPO_DIR / "scripts"
sys.path.insert(0, str(SCRIPTS_DIR))

DATALIST = Path(os.environ["T2G_LUMIERE_DATALIST"])
OUTPUT_DIR = Path(os.environ.get("T2G_OUTPUT_DIR", REPO_DIR / "runs" / "lumiere_eval")).resolve()
CACHE_DIR  = os.environ.get("T2G_HF_CACHE", None)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SPLIT       = os.environ.get("T2G_SPLIT", "validation")
TEXT_FIELD  = os.environ.get("T2G_TEXT_FIELD", "findings")
STEPS       = int(os.environ.get("T2G_STEPS", 200))
SEED        = int(os.environ.get("T2G_SEED", 42))
DEVICE_ARG  = os.environ.get("T2G_DEVICE", "cuda")
SCALE_OVR   = os.environ.get("T2G_SCALE_FACTOR")
SCALE_OVR   = float(SCALE_OVR) if SCALE_OVR not in (None, "") else None
NO_CHAN_REORDER = bool(int(os.environ.get("T2G_NO_CHANNEL_REORDER", 1)))
SAVE_NIFTI  = bool(int(os.environ.get("T2G_SAVE_NIFTI", 1)))
NUM_CASES   = int(os.environ.get("T2G_NUM_CASES", 0)) or None  # 0 = all
START_INDEX = int(os.environ.get("T2G_START_INDEX", 0))
CFG_LIST = [float(x) for x in os.environ.get("T2G_CFG_LIST", "1.0,2.0,4.5,7.0").split(",")]

MODELS = {
    "pinaya_ft": {
        "stage1_config": Path(os.environ["T2G_PINAYA_STAGE1_CFG"]),
        "ldm_config":    Path(os.environ["T2G_PINAYA_LDM_CFG"]),
        "stage1_uri":    Path(os.environ["T2G_PINAYA_STAGE1_URI"]),
        "ldm_ckpt":      Path(os.environ["T2G_PINAYA_LDM_CKPT"]),
    },
    "kl1e6_overfit": {
        "stage1_config": Path(os.environ["T2G_KL1E6_STAGE1_CFG"]),
        "ldm_config":    Path(os.environ["T2G_KL1E6_LDM_CFG"]),
        "stage1_uri":    Path(os.environ["T2G_KL1E6_STAGE1_URI"]),
        "ldm_ckpt":      Path(os.environ["T2G_KL1E6_LDM_CKPT"]),
    },
}

print("OUTPUT_DIR =", OUTPUT_DIR)
print("DATALIST   =", DATALIST)
print("CFG_LIST   =", CFG_LIST)
print("NUM_CASES  =", NUM_CASES or "all")
for tag, m in MODELS.items():
    print(f"  [{tag}] {m['stage1_uri'].name}  +  {m['ldm_ckpt'].name}")

## 2. Shared helpers (reuse the offline sampler primitives)

In [ ]:
import sys, os
from pathlib import Path
_here = Path.cwd()
for _cand in (_here, _here / 'scripts', _here.parent, _here.parent / 'scripts'):
    if (_cand / 'offline_sample_stage2_compare.py').is_file():
        if str(_cand) not in sys.path:
            sys.path.insert(0, str(_cand))
        break
else:
    raise RuntimeError('offline_sample_stage2_compare.py not found near cwd=%s' % _here)

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nibabel as nib
from tqdm.auto import tqdm

from text2glioma.utils import (
    MODALITY_NAMES,
    get_model,
    load_config,
    load_text_encoder_and_tokenizer,
    prepare_mask_conditioning,
    stage1_ify,
)

import offline_sample_stage2_compare as offline
from offline_sample_stage2_compare import (
    _build_val_transform,
    _compute_per_channel_ssim,
    _encode_text,
    _extract_state_dict,
    _get_tensor_affine,
    _infer_stage1_latent_channels,
    _make_grid,
    _resolve_device,
    _sample_latent,
    _save_sample_nifti,
    _save_tensor_nifti_native,
)

torch.manual_seed(SEED)
device = _resolve_device(DEVICE_ARG)
print("device =", device)

with open(DATALIST) as f:
    datalist = json.load(f)
if SPLIT not in datalist or not datalist[SPLIT]:
    raise KeyError(f"split '{SPLIT}' missing or empty in {DATALIST}")
all_entries = datalist[SPLIT]
end_index = (START_INDEX + NUM_CASES) if NUM_CASES else len(all_entries)
indices = list(range(START_INDEX, min(end_index, len(all_entries))))
print(f"will evaluate {len(indices)} sessions from LUMIERE")

In [ ]:
def _load_model(spec: dict):
    """Construct and load (stage1, ldm, scheduler, tokenizer, text_encoder, meta).

    Mirrors the bootstrap logic of offline_sample_stage2_compare.main().
    """
    config        = load_config(str(spec["ldm_config"]))
    stage1_config = load_config(str(spec["stage1_config"]))

    s1_params = stage1_config.setdefault("model", {}).setdefault("params", {})
    inferred = _infer_stage1_latent_channels(str(spec["stage1_uri"]))
    if inferred is not None:
        s1_params["latent_channels"] = inferred
    if device.type != "cuda":
        s1_params["use_flash_attention"] = False

    stage1 = stage1_ify(get_model("AutoencoderKL", stage1_config,
                                  from_file=str(spec["stage1_uri"])))
    stage1 = stage1.to(device).eval()
    for p in stage1.parameters():
        p.requires_grad = False

    # Probe actual latent channel count via a forward pass through Stage1Wrapper.
    # This is the only reliable way to detect channel-wise Pinaya wrapping where
    # a 1-ch VAE encodes a 4-ch image to 4 * base_latent_channels.
    s1_lat_ch = None
    try:
        _probe_item = dict(all_entries[indices[0]])
        _probe_has_label = bool(_probe_item.get("label"))
        _probe_tx = _build_val_transform(channel_reorder=not NO_CHAN_REORDER,
                                         has_label=_probe_has_label)
        _probe_batch = _probe_tx(_probe_item)
        _probe_img = _probe_batch["image"].unsqueeze(0).to(device)
        with torch.no_grad():
            _probe_z = stage1(_probe_img)
        s1_lat_ch = int(_probe_z.shape[1])
        del _probe_img, _probe_z
    except Exception as _exc:
        print(f"[warn] stage1 probe failed ({_exc}); falling back to module attrs")

    if s1_lat_ch is None:
        if hasattr(stage1, "model") and hasattr(stage1.model, "latent_channels"):
            s1_lat_ch = int(stage1.model.latent_channels)
        elif hasattr(stage1, "model") and hasattr(stage1.model, "quant_conv_mu"):
            s1_lat_ch = int(stage1.model.quant_conv_mu.out_channels)
        else:
            s1_lat_ch = int(config.get("model", {}).get("latent_channels", 4))

    n_mask = int(config.get("mask", {}).get("num_classes", 4))
    mcfg = config.setdefault("model", {})
    params = mcfg.setdefault("params", {})
    mcfg["latent_channels"] = s1_lat_ch
    params["in_channels"]   = s1_lat_ch + n_mask
    params["out_channels"]  = s1_lat_ch

    model = get_model(mcfg.get("name", "DiffusionModelUNet"), config)
    ckpt = torch.load(str(spec["ldm_ckpt"]), map_location="cpu")
    model.load_state_dict(_extract_state_dict(ckpt), strict=True)
    model = model.to(device).eval()

    sch_name = config.get("scheduler", {}).get("name", "DDIMScheduler")
    sch_params = config.get("scheduler", {}).get("params", {})
    if sch_name == "DDIMScheduler":
        from generative.networks.schedulers import DDIMScheduler
        scheduler = DDIMScheduler(**sch_params)
    else:
        from generative.networks.schedulers import DDPMScheduler
        scheduler = DDPMScheduler(**sch_params)

    tok, enc = load_text_encoder_and_tokenizer(
        config["conditioning"], cache_dir=CACHE_DIR, local_files_only=True,
    )
    cml = config["conditioning"].get("max_length")
    if cml is not None:
        tok.model_max_length = cml
    enc = enc.to(device).eval()
    for p in enc.parameters():
        p.requires_grad = False

    return dict(stage1=stage1, ldm=model, scheduler=scheduler,
                tokenizer=tok, text_encoder=enc,
                stage1_latent_ch=s1_lat_ch, num_mask_classes=n_mask)

def _sample_one(mdl: dict, item: dict, cfg_val: float, seed_offset: int):
    """Run one (case, CFG) sample. Returns (x_hat[1,C,H,W,D], image[1,C,H,W,D], scale_factor)."""
    has_label = bool(item.get("label"))
    transform = _build_val_transform(channel_reorder=not NO_CHAN_REORDER, has_label=has_label)
    batch = transform(dict(item))
    image = batch["image"].unsqueeze(0).to(device)
    label = batch["label"].unsqueeze(0).to(device) if has_label else None

    prompt = item.get(TEXT_FIELD) or item.get(
        "findings" if TEXT_FIELD == "impression" else "impression", ""
    )
    with torch.no_grad():
        z = mdl["stage1"](image)
        sf = (1.0 / max(z.std().item(), 1e-8)) if SCALE_OVR is None else SCALE_OVR
        latent_spatial = z.shape[2:]

        gen = torch.Generator(device=device).manual_seed(SEED + seed_offset)
        latent0 = torch.randn((1, mdl["stage1_latent_ch"]) + latent_spatial,
                              device=device, generator=gen)

        if label is not None:
            mask_cond = prepare_mask_conditioning(
                labels=label, latent_shape=latent_spatial,
                num_classes=mdl["num_mask_classes"], dropout_p=0.0,
            ).to(device)
        else:
            mask_cond = torch.zeros((1, mdl["num_mask_classes"]) + latent_spatial, device=device)
        mask_uncond = torch.zeros_like(mask_cond)

        cond   = _encode_text(mdl["tokenizer"], mdl["text_encoder"], str(prompt), device)
        uncond = _encode_text(mdl["tokenizer"], mdl["text_encoder"], "", device)

        mdl["scheduler"].set_timesteps(min(STEPS, mdl["scheduler"].num_train_timesteps))
        latent_cond = _sample_latent(
            mdl["ldm"], mdl["scheduler"], latent0, mask_cond, cond, device,
            uncond_embeds=uncond, uncond_mask_cond=mask_uncond,
            guidance_scale=float(cfg_val),
        )
        x_hat = mdl["stage1"].decode(latent_cond / sf).float().clamp(0, 1)
    return x_hat, image, sf, batch

## 3. Sweep — both models × all CFG × all cases

Each generated volume is saved as NIfTI under
`<OUTPUT_DIR>/<model_tag>/cfg_<X>p<Y>/sample_cond_<NNNN>.nii.gz` so
downstream FID / W1 / radiologist-read cells can re-use them without
re-sampling.

Per-case SSIM is recorded immediately.

In [ ]:
records = []
for tag, spec in MODELS.items():
    print(f"\n=== loading {tag} ===")
    mdl = _load_model(spec)
    model_out = OUTPUT_DIR / tag
    model_out.mkdir(parents=True, exist_ok=True)
    for case_i in tqdm(indices, desc=f"{tag} cases"):
        item = all_entries[case_i]
        subj = item.get("subject_id") or item.get("subject") or f"idx_{case_i}"
        for cfg_val in CFG_LIST:
            cfg_tag = f"cfg_{cfg_val:.2f}".replace(".", "p")
            cfg_dir = model_out / cfg_tag
            cfg_dir.mkdir(parents=True, exist_ok=True)
            out_nifti = cfg_dir / f"sample_cond_{case_i:04d}.nii.gz"
            if out_nifti.exists():
                # Recompute SSIM from the cached NIfTI to avoid resampling.
                arr = nib.load(str(out_nifti)).get_fdata().astype(np.float32)
                # _save_sample_nifti writes (X,Y,Z,C) using the source affine,
                # so to compute SSIM we re-run the val transform on a dict
                # that points at the cached NIfTI.
                stub = dict(item); stub["image"] = str(out_nifti)
                has_label = bool(item.get("label"))
                tr = _build_val_transform(channel_reorder=not NO_CHAN_REORDER, has_label=has_label)
                bx = tr(stub)
                x_hat = bx["image"].unsqueeze(0).to(device)
                # original
                tr_orig = _build_val_transform(channel_reorder=not NO_CHAN_REORDER, has_label=has_label)
                bo = tr_orig(dict(item))
                image = bo["image"].unsqueeze(0).to(device)
                ssim_per_ch, ssim_mean = _compute_per_channel_ssim(x_hat.float(), image.float(), MODALITY_NAMES)
                sf = float("nan")
            else:
                x_hat, image, sf, batch = _sample_one(mdl, item, cfg_val, seed_offset=case_i)
                if SAVE_NIFTI:
                    _save_sample_nifti(x_hat, item["image"], out_nifti)
                ssim_per_ch, ssim_mean = _compute_per_channel_ssim(x_hat, image, MODALITY_NAMES)

            for mod_name, val in ssim_per_ch.items():
                records.append(dict(model=tag, case_idx=case_i, subject_id=subj,
                                    cfg=float(cfg_val), modality=mod_name,
                                    ssim=float(val), scale_factor=sf))
            records.append(dict(model=tag, case_idx=case_i, subject_id=subj,
                                cfg=float(cfg_val), modality="mean",
                                ssim=float(ssim_mean), scale_factor=sf))
    # Release GPU memory before loading the next model.
    del mdl
    torch.cuda.empty_cache() if device.type == "cuda" else None

df = pd.DataFrame.from_records(records)
df.to_csv(OUTPUT_DIR / "ssim_vs_cfg_lumiere.csv", index=False)
print("\nSaved:", OUTPUT_DIR / "ssim_vs_cfg_lumiere.csv")
df.head()

## 4. SSIM vs CFG — both models on the same axes

In [ ]:
agg = df.groupby(["model", "cfg", "modality"])["ssim"].agg(["mean", "std", "count"]).reset_index()
agg.to_csv(OUTPUT_DIR / "ssim_vs_cfg_lumiere_agg.csv", index=False)

fig, ax = plt.subplots(figsize=(7, 4.5), dpi=150)
for tag in MODELS.keys():
    sub = agg[(agg["model"] == tag) & (agg["modality"] == "mean")].sort_values("cfg")
    ax.errorbar(sub["cfg"], sub["mean"], yerr=sub["std"].fillna(0.0),
                marker="o", lw=2.0, capsize=3, label=tag)
ax.set_xlabel("CFG scale"); ax.set_ylabel("mean SSIM (LUMIERE OOD)")
ax.set_title(f"LUMIERE OOD — mean SSIM, N={df['case_idx'].nunique()} sessions")
ax.grid(True, alpha=0.3); ax.legend()
fig.tight_layout(); fig.savefig(OUTPUT_DIR / "ssim_vs_cfg_lumiere.png", dpi=200)
plt.show()

## 5. Paired statistics — Pinaya FT vs KL=1e-6, per (modality, CFG)

Wilcoxon signed-rank on the per-case SSIM (paired by case). Cliff's δ as
the effect-size complement (insensitive to outliers).

Same cell can be re-run after FID/W1 are added by swapping the value column.

In [ ]:
from scipy.stats import wilcoxon

def _cliffs_delta(a: np.ndarray, b: np.ndarray) -> float:
    a, b = np.asarray(a), np.asarray(b)
    if a.size == 0 or b.size == 0:
        return float("nan")
    n = a.size * b.size
    return float(((a[:, None] > b[None, :]).sum() - (a[:, None] < b[None, :]).sum()) / n)

def paired_table(df_, value="ssim", model_a="pinaya_ft", model_b="kl1e6_overfit"):
    rows = []
    for (mod, cfg), sub in df_.groupby(["modality", "cfg"]):
        a = sub[sub["model"] == model_a].set_index("case_idx")[value]
        b = sub[sub["model"] == model_b].set_index("case_idx")[value]
        common = a.index.intersection(b.index)
        if len(common) < 5:
            continue
        av, bv = a.loc[common].values, b.loc[common].values
        try:
            w, p = wilcoxon(av, bv, zero_method="wilcox", alternative="two-sided")
        except ValueError:
            w, p = float("nan"), float("nan")
        rows.append(dict(modality=mod, cfg=float(cfg), n=len(common),
                         median_diff=float(np.median(av - bv)),
                         wilcoxon_W=float(w), wilcoxon_p=float(p),
                         cliffs_delta=_cliffs_delta(av, bv),
                         metric=value))
    return pd.DataFrame(rows).sort_values(["modality", "cfg"]).reset_index(drop=True)

paired_ssim = paired_table(df, value="ssim")
paired_ssim.to_csv(OUTPUT_DIR / "paired_stats_ssim_lumiere.csv", index=False)
paired_ssim

## 6. OOD degradation — LUMIERE vs internal validation

Loads the cached internal-validation SSIM CSVs (produced by the two
in-distribution notebooks) and reports the modality-wise SSIM degradation
ΔSSIM = SSIM_internal − SSIM_LUMIERE per (model, modality, CFG).

In [ ]:
INT_SSIM_CSVS = {
    "pinaya_ft":     os.environ.get("T2G_INT_SSIM_PINAYA", ""),
    "kl1e6_overfit": os.environ.get("T2G_INT_SSIM_KL1E6", ""),
}
delta_rows = []
for tag, path in INT_SSIM_CSVS.items():
    if not path or not Path(path).is_file():
        print(f"  (skip {tag}: internal SSIM CSV not found at {path})")
        continue
    int_df = pd.read_csv(path)
    int_mean = int_df.groupby(["cfg", "modality"])["ssim"].mean().reset_index().rename(columns={"ssim": "ssim_internal"})
    ood_mean = df[df["model"] == tag].groupby(["cfg", "modality"])["ssim"].mean().reset_index().rename(columns={"ssim": "ssim_lumiere"})
    merged = int_mean.merge(ood_mean, on=["cfg", "modality"], how="inner")
    merged["delta_ssim"] = merged["ssim_internal"] - merged["ssim_lumiere"]
    merged["model"] = tag
    delta_rows.append(merged)

if delta_rows:
    delta_df = pd.concat(delta_rows, ignore_index=True)
    delta_df.to_csv(OUTPUT_DIR / "ood_degradation_ssim.csv", index=False)
    print("Saved:", OUTPUT_DIR / "ood_degradation_ssim.csv")
    print(delta_df.head())

    # Bar plot per modality at CFG=1.0
    cfg_for_plot = 1.0
    sub = delta_df[delta_df["cfg"] == cfg_for_plot]
    fig, ax = plt.subplots(figsize=(6, 4), dpi=150)
    width = 0.35
    mods_sorted = list(MODALITY_NAMES) + ["mean"]
    x = np.arange(len(mods_sorted))
    for i, tag in enumerate(MODELS.keys()):
        vals = [float(sub[(sub["model"] == tag) & (sub["modality"] == m)]["delta_ssim"].mean()) if not sub[(sub["model"] == tag) & (sub["modality"] == m)].empty else np.nan for m in mods_sorted]
        ax.bar(x + (i - 0.5) * width, vals, width, label=tag)
    ax.set_xticks(x); ax.set_xticklabels(mods_sorted)
    ax.set_ylabel("ΔSSIM (internal − LUMIERE)")
    ax.set_title(f"OOD degradation at CFG={cfg_for_plot}")
    ax.axhline(0, color="k", lw=0.5); ax.legend(); ax.grid(True, alpha=0.3, axis="y")
    fig.tight_layout(); fig.savefig(OUTPUT_DIR / "ood_degradation_ssim.png", dpi=200)
    plt.show()
else:
    print("No internal SSIM CSVs supplied — set T2G_INT_SSIM_PINAYA / T2G_INT_SSIM_KL1E6.")

## 7. FID on LUMIERE (uses the multi-channel fix from 2026-06-28)

Re-uses the cached NIfTIs saved above. Mirrors the FID cell in the
in-distribution notebooks. The 3D autoencoder path uses each model's
*own* stage-1 encoder as the feature extractor (in-domain FID).

In [ ]:
import compute_fid
importlib.reload(compute_fid)
from compute_fid import FIDAccumulator

FID_MODE    = os.environ.get("T2G_FID_MODE", "3d_medicalnet")
FID_WEIGHTS = os.environ.get("T2G_FID_WEIGHTS")  # MedicalNet / RadImageNet weights
FID_SLICES  = int(os.environ.get("T2G_FID_SLICES", 12))
ALL_KEY = "all"  # bucket name for the 3d_autoencoder mode

fid_rows = []
for tag in MODELS.keys():
    # In 3d_autoencoder mode we must use this model's own stage1 as the feature extractor.
    if FID_MODE == "3d_autoencoder":
        mdl = _load_model(MODELS[tag]); s1 = mdl["stage1"]
    else:
        s1 = None
    fid = FIDAccumulator(mode=FID_MODE, device=device,
                         backbone_weights=Path(FID_WEIGHTS) if FID_WEIGHTS else None,
                         stage1=s1, slices_per_volume=FID_SLICES)
    model_out = OUTPUT_DIR / tag
    for case_i in tqdm(indices, desc=f"FID {tag}"):
        item = all_entries[case_i]
        if not item.get("label"):
            continue
        real_arr = nib.load(item["image"]).get_fdata().astype(np.float32)
        if real_arr.ndim == 3:
            real_arr = real_arr[..., None]
        real_arr = np.moveaxis(real_arr, -1, 0)
        mask_arr = nib.load(item["label"]).get_fdata()
        mask_t = torch.from_numpy((mask_arr > 0).astype(bool))

        if FID_MODE == "3d_autoencoder":
            fid.add_real(torch.from_numpy(real_arr), mask_t, ALL_KEY)
        else:
            for c, mod in enumerate(MODALITY_NAMES):
                if c >= real_arr.shape[0]: continue
                fid.add_real(torch.from_numpy(real_arr[c]), mask_t, mod)

        for cfg_val in CFG_LIST:
            cfg_tag = f"cfg_{cfg_val:.2f}".replace(".", "p")
            p = model_out / cfg_tag / f"sample_cond_{case_i:04d}.nii.gz"
            if not p.exists():
                continue
            gen_arr = nib.load(str(p)).get_fdata().astype(np.float32)
            if gen_arr.ndim == 3: gen_arr = gen_arr[..., None]
            gen_arr = np.moveaxis(gen_arr, -1, 0)
            if FID_MODE == "3d_autoencoder":
                if gen_arr.shape[0] != real_arr.shape[0]:
                    fixed = np.zeros((real_arr.shape[0],) + gen_arr.shape[1:], dtype=gen_arr.dtype)
                    fixed[:min(gen_arr.shape[0], real_arr.shape[0])] = gen_arr[:min(gen_arr.shape[0], real_arr.shape[0])]
                    gen_arr = fixed
                fid.add_gen(torch.from_numpy(gen_arr), mask_t, ALL_KEY, cfg=cfg_val)
            else:
                for c, mod in enumerate(MODALITY_NAMES):
                    if c >= gen_arr.shape[0]: continue
                    fid.add_gen(torch.from_numpy(gen_arr[c]), mask_t, mod, cfg=cfg_val)

    fid_df = fid.compute(); fid_df["model"] = tag
    fid_rows.append(fid_df)
    if FID_MODE == "3d_autoencoder":
        del mdl; torch.cuda.empty_cache() if device.type == "cuda" else None

fid_all = pd.concat(fid_rows, ignore_index=True) if fid_rows else pd.DataFrame()
if not fid_all.empty:
    fid_csv = OUTPUT_DIR / f"fid_vs_cfg_lumiere_{FID_MODE}.csv"
    fid_all.to_csv(fid_csv, index=False)
    print("Saved:", fid_csv)
fid_all

## 8. Qualitative panel — one row per case, two columns per CFG

Real | Pinaya FT | KL1e6 — for a small subset of cases.

In [ ]:
QUAL_CFG = float(os.environ.get("T2G_QUAL_CFG", 1.0))
QUAL_N   = int(os.environ.get("T2G_QUAL_N", 4))
qual_indices = indices[:QUAL_N]

cfg_tag = f"cfg_{QUAL_CFG:.2f}".replace(".", "p")
for case_i in qual_indices:
    item = all_entries[case_i]
    has_label = bool(item.get("label"))
    tr = _build_val_transform(channel_reorder=not NO_CHAN_REORDER, has_label=has_label)
    bo = tr(dict(item)); image_cpu = bo["image"].unsqueeze(0)
    depth = image_cpu.shape[-1]
    depth_idx = [depth // 4, depth // 2, (3 * depth) // 4]
    real_grid = _make_grid(image_cpu, depth_idx)

    panels = [("REAL", real_grid)]
    for tag in MODELS.keys():
        gp = OUTPUT_DIR / tag / cfg_tag / f"sample_cond_{case_i:04d}.nii.gz"
        if not gp.exists(): continue
        stub = dict(item); stub["image"] = str(gp)
        tr2 = _build_val_transform(channel_reorder=not NO_CHAN_REORDER, has_label=has_label)
        b2 = tr2(stub); x_hat = b2["image"].unsqueeze(0)
        panels.append((tag, _make_grid(x_hat, depth_idx)))

    fig, axes = plt.subplots(len(panels), 1, dpi=200, figsize=(10, 2.5 * len(panels)))
    if len(panels) == 1: axes = [axes]
    for ax, (name, g) in zip(axes, panels):
        ax.imshow(g, cmap="gray"); ax.set_title(f"{name}  (case {case_i})"); ax.axis("off")
        n_ch = image_cpu.shape[1]
        w_per_ch = g.shape[1] / n_ch
        for c in range(n_ch):
            mod = MODALITY_NAMES[c] if c < len(MODALITY_NAMES) else f"ch{c}"
            ax.text(int(w_per_ch * c) + 2, 10, mod, fontsize=6, color="yellow")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f"qualitative_lumiere_{case_i:04d}_{cfg_tag}.png", dpi=200)
    plt.show()